In [0]:
import dlt

In [0]:
from pyspark.sql.functions import col, current_timestamp
@dlt.table(
    name = "bronze_addresses",
    comment = "Raw data from the addresses table in the database.",
    table_properties = {"quality": "bronze"}
)
def create_bronze_addresses():
  return (
    spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", "/Volumes/circuitbox/lakehouse/_schemas/bronze_addresses_schema")
      .option("cloudFiles.inferColumnTypes", "true")
      .load('/Volumes/circuitbox/landing/operational_data/addresses/')
      .withColumn("input_file_path", col("_metadata.file_path"))
      .withColumn("created_date", current_timestamp())
  )

In [0]:
@dlt.table(
    name = "silver_addressess_clean",
    comment = "Data from the addresses table from the bronze layer.",
    table_properties = {"quality": "silver"}
)
@dlt.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_address_line_1", "address_line_1 IS NOT NULL")
@dlt.expect("valid_postcode","length(postcode) = 5")

def create_silver_addressess_clean():
  return (
    spark.readStream.table("LIVE.bronze_addresses").select(
        "customer_id",
        "address_line_1", 
        "city", 
        "state", 
        "postcode", 
        col("created_date").cast("date")
  )
)

In [0]:
dlt.create_streaming_table(
  name="silver_addressess",
    comment="Data from the cleaned addresses table from the silver layer.",
    table_properties={"quality": "silver"}
)
dlt.apply_changes(
    target = "silver_addressess",
    source = "silver_addressess_clean",
    keys = ["customer_id"],
    sequence_by = col("created_date"),
    stored_as_scd_type = 2
)